# Fine-tune ViHealthBERT cho NER trên VietMed-NER

Notebook này sử dụng siêu tham số tốt nhất từ quá trình tìm kiếm:
- **Base model**: demdecuong/vihealthbert-base-syllable
- **Learning rate**: 3e-05
- **Epochs**: 8
- **Weight decay**: 0.01
- **Seed**: 2024
- **Validation F1**: 83.29%

Chỉ cần upload lên Google Colab và nhấn Run All để tạo ra mô hình fine-tuned.

In [ ]:
# Cài đặt thư viện cần thiết
!pip install -q --upgrade pip setuptools wheel
!pip install -q transformers datasets accelerate
!pip install -q seqeval --no-build-isolation

In [ ]:
# Import thư viện
import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from seqeval.metrics import f1_score, precision_score, recall_score

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Siêu tham số tốt nhất
BASE_MODEL = 'demdecuong/vihealthbert-base-syllable'
LEARNING_RATE = 3e-05
EPOCHS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 2024
MAX_LENGTH = 256
BATCH_SIZE = 16
OUTPUT_DIR = './vihealthbert-vietmed-ner-final'

print(f'Base model: {BASE_MODEL}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Epochs: {EPOCHS}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'Seed: {SEED}')

In [ ]:
# Tải dataset VietMed-NER
dataset = load_dataset('leduckhai/VietMed-NER')

# Bỏ cột audio để tối ưu bộ nhớ
if 'audio' in dataset['train'].column_names:
    dataset = dataset.remove_columns('audio')

print('Dataset splits:')
for split in dataset:
    print(f'  {split}: {len(dataset[split])} examples')

In [ ]:
# Chuẩn bị label vocabulary
label_column = 'labels' if 'labels' in dataset['train'].column_names else 'tags'
feature = dataset['train'].features[label_column].feature

if hasattr(feature, 'names'):
    label_names = list(feature.names)
else:
    label_names = sorted({
        str(label)
        for example in dataset['train']
        for label in example[label_column]
    })

label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

print(f'Number of labels: {len(label_names)}')
print(f'Labels: {label_names[:10]}...')

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.model_max_length = MAX_LENGTH

print(f'Tokenizer: {tokenizer.__class__.__name__}')
print(f'Max length: {MAX_LENGTH}')

In [ ]:
# Hàm tokenize và align labels
def label_to_id(label):
    if isinstance(label, int) and hasattr(feature, 'names'):
        label = feature.int2str(label)
    return label2id[str(label)]

def encode_words(words, row_labels=None):
    raw_ids = []
    raw_word_ids = []
    raw_labels = [] if row_labels is not None else None
    
    for word_idx, word in enumerate(words):
        subtoken_ids = tokenizer.encode(word, add_special_tokens=False)
        if not subtoken_ids:
            continue
        raw_ids.extend(subtoken_ids)
        raw_word_ids.extend([word_idx] * len(subtoken_ids))
        if raw_labels is not None:
            raw_labels.extend(
                [label_to_id(row_labels[word_idx])] + [-100] * (len(subtoken_ids) - 1)
            )
    
    input_ids = tokenizer.build_inputs_with_special_tokens(raw_ids)
    
    # Tìm vị trí bắt đầu của raw tokens
    raw_start = None
    for start in range(len(input_ids) - len(raw_ids) + 1):
        if input_ids[start:start + len(raw_ids)] == raw_ids:
            raw_start = start
            break
    
    if raw_start is None:
        raise ValueError('Could not locate raw tokens')
    
    # Truncate nếu quá dài
    input_ids = input_ids[:MAX_LENGTH]
    raw_end = min(raw_start + len(raw_ids), len(input_ids))
    
    # Align labels
    aligned_word_ids = [None] * len(input_ids)
    aligned_labels = [-100] * len(input_ids) if raw_labels is not None else None
    
    for raw_idx, pos in enumerate(range(raw_start, raw_end)):
        aligned_word_ids[pos] = raw_word_ids[raw_idx]
        if aligned_labels is not None:
            aligned_labels[pos] = raw_labels[raw_idx]
    
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': aligned_labels
    }

def tokenize_and_align(examples):
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for words, row_labels in zip(examples['words'], examples[label_column]):
        encoded = encode_words(words, row_labels)
        for key in encoded:
            batch[key].append(encoded[key])
    return batch

print('Tokenizing datasets...')
tokenized_train = dataset['train'].map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset['train'].column_names,
)
tokenized_val = dataset['validation'].map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset['validation'].column_names,
)

print(f'Train: {len(tokenized_train)} examples')
print(f'Validation: {len(tokenized_val)} examples')

In [ ]:
# Hàm compute metrics — tương thích các phiên bản seqeval

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predicted_ids = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label_row in zip(predicted_ids, labels):
        pred_tags = []
        gold_tags = []
        for pred_id, gold_id in zip(prediction, label_row):
            if gold_id == -100:
                continue
            pred_tags.append(id2label[int(pred_id)])
            gold_tags.append(id2label[int(gold_id)])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)

    return {
        'precision': float(precision_score(true_labels, true_predictions)),
        'recall': float(recall_score(true_labels, true_predictions)),
        'f1': float(f1_score(true_labels, true_predictions)),
    }

print('Metrics function ready')

In [ ]:
# Load model
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print(f'Model loaded: {BASE_MODEL}')
print(f'Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

In [ ]:
# Training arguments — tương thích nhiều phiên bản Transformers
import inspect

training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='linear',
    warmup_steps=int(0.1 * EPOCHS * (len(tokenized_train) // BATCH_SIZE)),
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to='none',
)

# Transformers mới đổi evaluation_strategy thành eval_strategy.
training_parameters = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in training_parameters:
    training_kwargs['eval_strategy'] = 'epoch'
else:
    training_kwargs['evaluation_strategy'] = 'epoch'

# Chỉ bật bf16 khi GPU Colab hỗ trợ; tránh lỗi trên T4.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    training_kwargs['bf16'] = True

training_args = TrainingArguments(**training_kwargs)
print('Training arguments configured')
print('Evaluation argument:', 'eval_strategy' if 'eval_strategy' in training_parameters else 'evaluation_strategy')

In [ ]:
# Create Trainer — tương thích nhiều phiên bản Transformers
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer_parameters = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
print('Trainer created')

In [ ]:
# Train model
print('Starting training...')
train_result = trainer.train()
print('Training completed!')
print(f'Training time: {train_result.metrics["train_runtime"]:.2f} seconds')

In [ ]:
# Evaluate on validation set
print('Evaluating on validation set...')
eval_results = trainer.evaluate()
print('\nValidation Results:')
print(f'  Loss: {eval_results["eval_loss"]:.4f}')
print(f'  Precision: {eval_results["eval_precision"]:.4f}')
print(f'  Recall: {eval_results["eval_recall"]:.4f}')
print(f'  F1: {eval_results["eval_f1"]:.4f}')

In [ ]:
# Save final model
final_model_dir = f'{OUTPUT_DIR}/final'
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print(f'\nModel saved to: {final_model_dir}')

# Save config
config_info = {
    'base_model': BASE_MODEL,
    'learning_rate': LEARNING_RATE,
    'epochs': EPOCHS,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'seed': SEED,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'validation_f1': eval_results['eval_f1'],
    'validation_precision': eval_results['eval_precision'],
    'validation_recall': eval_results['eval_recall'],
}

with open(f'{final_model_dir}/training_config.json', 'w', encoding='utf-8') as f:
    json.dump(config_info, f, indent=2, ensure_ascii=False)

print('Training config saved')
print('\n=== DONE ===')

## Test mô hình vừa train

Chạy cell dưới để test mô hình trên một câu mẫu.

In [ ]:
# Test với một câu mẫu
from transformers import pipeline

# Load model vừa train
ner_pipeline = pipeline(
    'ner',
    model=final_model_dir,
    tokenizer=final_model_dir,
    aggregation_strategy='simple'
)

# Test câu
test_sentence = "Bệnh nhân bị đau đầu và sốt cao, đã uống paracetamol."
results = ner_pipeline(test_sentence)

print(f'Test sentence: {test_sentence}')
print('\nDetected entities:')
for entity in results:
    print(f"  {entity['word']}: {entity['entity_group']} (score: {entity['score']:.3f})")